In [11]:
import pandas as pd
from sklearn.manifold import MDS
from irina import utility_functions as uf
from test_clustering import df_top_subfields

In [34]:
path = "../data/"
df_distance = pd.read_csv(path+"df_country_dist_cosine_world_log.csv", index_col=0)
df_country_subfield = pd.read_csv(path+"df_country_subfield_norm_world_log.csv", index_col=0)

In [35]:
df_country_stats = (
    pd.read_csv(path+"df_country_stats.csv")
    .assign(share_articles=lambda df: df.total_articles / df.total_articles.sum())
)
df_country_stats

,country,entropy,norm_entropy,total_articles,log_total_articles,gini,share_articles
0,US,4.831009,0.873690,590234.0,13.288274,0.613181,2.120605e-01
1,CN,4.552830,0.823382,566541.0,13.247305,0.702698,2.035481e-01
2,GB,4.839565,0.875238,156317.0,11.959641,0.613604,5.616190e-02
3,DE,4.745337,0.858197,134641.0,11.810367,0.645080,4.837410e-02
4,FR,4.747294,0.858551,121503.0,11.707694,0.642908,4.365386e-02
...,...,...,...,...,...,...,...
215,GQ,0.693147,0.125356,2.0,0.693147,0.000000,7.185643e-07
216,SX,-0.000000,-0.000000,1.0,0.000000,0.000000,3.592821e-07
217,SB,-0.000000,-0.000000,1.0,0.000000,0.000000,3.592821e-07
218,MP,-0.000000,-0.000000,1.0,0.000000,0.000000,3.592821e-07


In [49]:
df_top_subfield = (
    df_country_subfield
    .assign(top1_subfield=lambda df: df.apply(lambda row: int(df.columns[np.where(row == row.max())[0][0]]), axis=1))
    .reset_index(names=["country"])
    [["country", "top1_subfield"]]
    .merge(uf.df_topics[["subfield_id", "subfield_name", "field_name", "domain_name"]].drop_duplicates(),
           left_on="top1_subfield",
           right_on="subfield_id",
           how="left")
)

In [52]:
df_country_info = (
    df_country_stats
    .merge(df_top_subfield, left_on="country", right_on="country")
)

# MDS for distances

In [ ]:
mds = MDS(
    n_components=3,
    dissimilarity="precomputed",
    random_state=42,
    n_init=4,
    max_iter=300
)

coords = mds.fit_transform(df_distance.values)  # shape (N, 3)

In [61]:
sizes = 10 + 200 * df_country_info.share_articles

color_by_field = {
    "Social Sciences": "red",
    "Health Sciences": "green",
    "Physical Sciences": "blue",
    "Life Sciences": "purple",
}

colors = [color_by_field[f] for f in df_country_info.domain_name]

In [62]:
df_country_info[["country", "field_name", "share_articles"]]

,country,field_name,share_articles
0,US,"Biochemistry, Genetics and Molecular Biology",2.120605e-01
1,CN,Engineering,2.035481e-01
2,GB,Social Sciences,5.616190e-02
3,DE,Health Professions,4.837410e-02
4,FR,"Business, Management and Accounting",4.365386e-02
...,...,...,...
215,GQ,Medicine,7.185643e-07
216,SX,Medicine,3.592821e-07
217,SB,"Business, Management and Accounting",3.592821e-07
218,MP,Environmental Science,3.592821e-07


In [63]:
import plotly.graph_objects as go

fig = go.Figure(
    data=go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers",
        marker=dict(
            size=sizes,
            color=colors,
            opacity=0.7,
            line=dict(width=0.5, color="black")
        ),
        text=[
            f"Country: {c}<br>"
            f"Field: {f}<br>"
            f"Share: {s:.3%}"
            for c, f, s in df_country_info[["country", "field_name", "share_articles"]].values
        ],
        hoverinfo="text"
    )
)

fig.update_layout(
    title="3D layout of countries based on distance matrix",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    height=800
)

fig.show()

# UMAP on vectors

In [91]:
import umap
import numpy as np

X = df_country_subfield.fillna(0).values  # shape (n, 252)

reducer = umap.UMAP(
    n_components=3,
    metric="cosine",
    n_neighbors=3,
    min_dist=0.1,
    random_state=42
)

X_3d = reducer.fit_transform(X)

/Users/irinavorobeva/PycharmProjects/geoscience/.venv/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [92]:
import pandas as pd

df_plot = pd.DataFrame({
    "x": X_3d[:, 0],
    "y": X_3d[:, 1],
    "z": X_3d[:, 2],
    "share": df_country_info.share_articles,
    "field": df_country_info.field_name,
    "label": df_country_info.country,
    "domain": df_country_info.domain_name
})

In [93]:
color_by_domain = {
    "Social Sciences": "#e41a1c",
    "Health Sciences": "#4daf4a",
    "Physical Sciences": "#377eb8",
    "Life Sciences": "#984ea3",
}

In [94]:
import plotly.graph_objects as go

fig = go.Figure()

for domain, color in color_by_domain.items():
    dff = df_plot[df_plot["domain"] == domain]

    fig.add_trace(
        go.Scatter3d(
            x=dff["x"],
            y=dff["y"],
            z=dff["z"],
            mode="markers",
            # name=dff["field"],
            marker=dict(
                size=dff["share"],
                sizemode="area",
                sizeref=2. * dff["share"].max() / (30. ** 2),
                color=color,
                opacity=0.7,
                line=dict(width=0)
            ),
            # hovertemplate=[
            #     "<b>%{text}</b><br>"
            #     "Field: " + field + "<br>"
            #     "Share: %{marker.size:.3f}<extra></extra>"
            # for field in dff["field"]],
            text=dff["label"]
        )
    )

fig.update_layout(
    title="3D UMAP of Topics / Countries",
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
        bgcolor="white"
    ),
    legend_title="Dominant field",
    height=800
)

fig.show()